# ISLP Chapter 2 — Ingestion Walkthrough

This notebook walks through the hybrid ingestion pipeline for ISLP Chapter 2. The pipeline combines a **deterministic regex pass** (headers, page markers, formulas, images, tables) with an **LLM enrichment pass** (synopsis + extended index terms). Outputs follow the **ParentDocumentRetriever** pattern: small child chunks are embedded into Chroma for retrieval, while their larger parent sections are stored separately and returned at query time for richer context.

## 0. Prereqs

Make sure the project venv is active, the `.env` file is populated (OpenAI key, Chroma host/port), and the Chroma container is running locally via `docker compose up` on port `8002`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.core.config import settings
print("Chroma:", settings.chroma_host, settings.chroma_port)
print("Embedding:", settings.embedding_model)
print("Default LLM provider:", settings.default_provider)

ModuleNotFoundError: No module named 'pydantic'

## 1. Static metadata (Stage A)

Each book has a small YAML config in `ingestion/books/` listing slug, authors, source path, and chapter line ranges. A Claude-extracted `book.json` (in `data/parsed/<slug>/`) supplies the global **index terms** and **bibliography**. `load_book_static_metadata` merges both into a `BookMetadata` object.

In [ ]:
from src.ingestion.pipeline import load_book_static_metadata
book = load_book_static_metadata("islp")
print(book.name, "—", ", ".join(book.authors))
print("Index terms:", len(book.index_terms))
print("Bibliography entries:", len(book.bibliography))
print("First 5 index terms:", book.index_terms[:5])

## 2. Regex pass (Stage B)

The regex pass slices the markdown source between the configured line range and emits one `SectionMetadata` per H2. It captures the fields described in `ingestion/reference.md`: H1/H2 hierarchy, page markers, LaTeX formulas, image references, and table references.

In [ ]:
from pathlib import Path
from src.ingestion import regex_pass
sections = regex_pass.parse_chapter(
    book_slug="islp", chapter_id="ch02",
    source_path=Path(book.source_path),
    line_start=650, line_end=2557,
)
print(f"Sections: {len(sections)}")
for s in sections[:5]:
    print(f"  {s.section_id:30s} chars={s.char_count} formulas={len(s.formulas)} imgs={len(s.images)}")

## 3. Inspect one section

Peek at what the regex pass produced for a single section: heading path, page span, captured formulas/images/table references, and a snippet of the raw text.

In [ ]:
s = sections[2]
print("h2_path:", s.h2_path)
print("pages:", s.page_from, "-", s.page_to)
print("formulas (first 3):", s.formulas[:3])
print("images:", s.images[:3])
print("table_refs:", s.table_refs[:2])
print("text preview:", s.text[:300], "...")

## 4. LLM enrichment (Stage C) — provider=OpenAI nano V5

Each section is sent to the configured LLM (here, the OpenAI nano model) which returns a one-paragraph **synopsis** and an **index_extended** list of section-local terms grounded against the global index. Results are cached, so re-running skips the API call on cache hits.

In [ ]:
from src.ingestion.llm_enrich import enrich_section
s_enriched = enrich_section(sections[2], provider="openai", existing_index=book.index_terms)
print("Synopsis:", s_enriched.synopsis)
print("Index extended:", s_enriched.index_extended)

## 5. Parent + child documents (Stage D)

`build_documents.build` turns enriched sections into the two-tier representation: **child** chunks (small, embedded into Chroma + BM25) and **parent** documents (full sections, kept in the docstore and returned at query time).

In [ ]:
from src.ingestion.build_documents import build
children, child_ids, parents = build(sections[:5])
print(f"parents={len(parents)}  children={len(children)}")
print("Example child metadata:", children[0].metadata)
print("Example parent length:", len(list(parents.values())[0].page_content))

## 6. Full pipeline via CLI

End-to-end, the pipeline is driven by a single command. It loads metadata, runs regex + LLM enrichment, persists children to Chroma and BM25, saves parents, and updates the manifest.

```bash
# From the project root:
python -m src.ingestion.pipeline --book islp --chapter ch02 --provider openai
python -m src.ingestion.pipeline --status
```

## 7. Read the manifest

The manifest at `data/parsed/manifest.json` tracks every successfully ingested chapter (with its chapter hash) so re-running the pipeline is idempotent.

In [ ]:
from src.ingestion import manifest
m = manifest.load()
for e in m.entries:
    print(e.book_slug, e.chapter_id, "→", e.chunk_count, "chunks,", e.parent_count, "parents,", e.status)